# TransCODE All-Tool Comparison: Annotated vs Non-Annotated ORFs

Compares annotated (CDS) and non-annotated (non_CDS) outputs for all five tools
(PRICE, RiboTIE, ORFQuant, iRibo, RibORF2) using `source_feature_class` from
the rebuilt translon DB.

RibORF2 classification uses reference_cds JOIN (set by `classify_unknown_feature_class`
during DB build).  All other tools use their native annotated/novel output split.


In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── DB path ────────────────────────────────────────────────────────────────────
DB = Path("/Users/jackt/translon_db/translon_db_v2/translon_db/translon_db/translons.sqlite")
# After HPC rebuild, update to the new path:
# DB = Path("/path/to/translon_db_rebuild/translon_db/translons.sqlite")
if not DB.exists():
    raise FileNotFoundError(f"DB not found: {DB}  — update DB path above")
con = sqlite3.connect(DB)
print(f"Opened: {DB}")

# ── visual constants ────────────────────────────────────────────────────────────
TOOL_ORDER  = ["PRICE", "RiboTIE", "ORFQuant", "iRibo", "RibORF2"]
CLS_ORDER   = ["cds", "non_cds"]
TOOL_COLORS = {
    "PRICE":    "#f58518",
    "RiboTIE":  "#54a24b",
    "ORFQuant": "#4c78a8",
    "iRibo":    "#e45756",
    "RibORF2":  "#9467bd",
}
CLS_COLORS = {"cds": "#2ca02c", "non_cds": "#1f77b4"}
CLS_LABELS = {"cds": "Annotated CDS", "non_cds": "Novel / non-CDS"}

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 120)


## DB overview

In [ ]:
# Parser manifest
manifest = pd.read_sql(
    "SELECT detected_tool, parser_name, parser_status, COUNT(*) AS files, SUM(translons) AS translons "
    "FROM parser_manifest GROUP BY detected_tool, parser_name, parser_status "
    "ORDER BY detected_tool, parser_name",
    con,
)
display(manifest)


In [ ]:
# Feature class breakdown per tool
cls_breakdown = pd.read_sql(
    "SELECT source_tool, source_feature_class, COUNT(*) AS n "
    "FROM translons GROUP BY source_tool, source_feature_class ORDER BY source_tool, source_feature_class",
    con,
)
display(cls_breakdown.pivot_table(index="source_tool", columns="source_feature_class",
                                   values="n", fill_value=0, aggfunc="sum"))


## CDS recall (exact coordinate match vs reference annotation)

In [ ]:
recall = pd.read_sql("SELECT * FROM cds_recall_by_tool ORDER BY source_tool", con)
display(recall)


## Main aggregation

Pull per-tool × per-class × per-sample counts from the DB using SQL.
Avoids loading the full 5 M-row translons table.


In [ ]:
# QC-passing translons only; exclude known FASTQ re-runs (PRICE pancreas_mymapping)
summary = pd.read_sql("""
    SELECT
        source_tool,
        sample_id,
        source_feature_class AS cls,
        start_codon_class,
        terminal_codon_class,
        block_count,
        spliced_length_nt,
        COUNT(*) AS n
    FROM translons
    WHERE qc_status = 'pass'
      AND sample_id NOT GLOB '*_fastq'
      AND source_feature_class IN ('cds', 'non_cds')
    GROUP BY source_tool, sample_id, source_feature_class,
             start_codon_class, terminal_codon_class, block_count, spliced_length_nt
""", con)

# Normalise: use just cds / non_cds labels
summary["cls"] = summary["cls"].str.lower().str.replace("-", "_")
print(f"Summary rows: {len(summary):,}")
summary.head(3)


## Plot 1 — Total ORF counts by tool and class

In [ ]:
totals = (
    summary.groupby(["source_tool", "cls"])["n"]
    .sum()
    .reset_index()
    .pivot_table(index="source_tool", columns="cls", values="n", fill_value=0)
    .reindex(index=TOOL_ORDER, columns=CLS_ORDER, fill_value=0)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# absolute counts
ax = axes[0]
x = np.arange(len(TOOL_ORDER))
w = 0.35
for i, cls in enumerate(CLS_ORDER):
    vals = totals[cls].values
    bars = ax.bar(x + i * w - w / 2, vals, width=w,
                  color=CLS_COLORS[cls], label=CLS_LABELS[cls], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(TOOL_ORDER, rotation=0)
ax.set_ylabel("Total ORFs (QC-pass)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e3:.0f}k"))
ax.legend(); ax.set_title("Total call space per tool")

# log scale to show small classes
ax = axes[1]
for i, cls in enumerate(CLS_ORDER):
    vals = totals[cls].values
    ax.bar(x + i * w - w / 2, vals, width=w,
           color=CLS_COLORS[cls], label=CLS_LABELS[cls], alpha=0.85)
ax.set_yscale("log")
ax.set_xticks(x); ax.set_xticklabels(TOOL_ORDER, rotation=0)
ax.set_ylabel("Total ORFs (QC-pass, log scale)")
ax.legend(); ax.set_title("Log scale — reveals small CDS class sizes")

plt.tight_layout()
plt.savefig("all_tool_call_space.png", dpi=180, bbox_inches="tight")
plt.show()
print(totals)


## Plot 2 — Per-sample call counts per tool (non-CDS)

In [ ]:
per_sample = (
    summary.groupby(["source_tool", "sample_id", "cls"])["n"]
    .sum()
    .reset_index()
)

non_cds_ps = per_sample[per_sample["cls"] == "non_cds"].copy()

fig, ax = plt.subplots(figsize=(10, 4.5))
rng = np.random.default_rng(42)
positions = {t: i for i, t in enumerate(TOOL_ORDER)}
for tool in TOOL_ORDER:
    vals = non_cds_ps.loc[non_cds_ps["source_tool"] == tool, "n"].values
    if len(vals) == 0:
        continue
    jitter = rng.uniform(-0.2, 0.2, len(vals))
    ax.scatter(positions[tool] + jitter, vals, color=TOOL_COLORS[tool],
               alpha=0.7, s=40, label=tool)
    ax.plot([positions[tool] - 0.3, positions[tool] + 0.3],
            [np.median(vals)] * 2, color=TOOL_COLORS[tool], lw=2)

ax.set_xticks(list(positions.values()))
ax.set_xticklabels(TOOL_ORDER)
ax.set_ylabel("Novel ORF calls per sample")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e3:.0f}k"))
ax.set_title("Per-sample novel (non-CDS) call space  |  horizontal line = median")
plt.tight_layout()
plt.savefig("all_tool_noncds_per_sample.png", dpi=180, bbox_inches="tight")
plt.show()


## Plot 3 — CDS recall and annotated ORF recovery

In [ ]:
cds_ps = per_sample[per_sample["cls"] == "cds"].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# left: CDS call count per tool per sample
ax = axes[0]
for tool in TOOL_ORDER:
    vals = cds_ps.loc[cds_ps["source_tool"] == tool, "n"].values
    if len(vals) == 0:
        ax.scatter([], [], color=TOOL_COLORS[tool], s=40, label=f"{tool} (no data)")
        continue
    jitter = rng.uniform(-0.2, 0.2, len(vals))
    ax.scatter(positions[tool] + jitter, vals,
               color=TOOL_COLORS[tool], alpha=0.7, s=40, label=tool)
    ax.plot([positions[tool] - 0.3, positions[tool] + 0.3],
            [np.median(vals)] * 2, color=TOOL_COLORS[tool], lw=2)
ax.set_xticks(list(positions.values()))
ax.set_xticklabels(TOOL_ORDER)
ax.set_ylabel("Annotated CDS calls per sample")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e3:.0f}k"))
ax.set_title("Per-sample annotated (CDS) call space")

# right: exact CDS recall (from DB table)
ax = axes[1]
recall_filt = recall[recall["source_tool"] != "ANY"].copy()
recall_filt = recall_filt.set_index("source_tool").reindex(TOOL_ORDER).reset_index()
colors = [TOOL_COLORS.get(t, "grey") for t in recall_filt["source_tool"]]
ax.bar(recall_filt["source_tool"], recall_filt["exact_cds_recall_pct"], color=colors, alpha=0.85)
ax.set_ylabel("Exact CDS recall (%)")
ax.set_title("Exact genomic-structure CDS recall vs reference")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig("all_tool_cds_recovery.png", dpi=180, bbox_inches="tight")
plt.show()


## Plot 4 — Start codon profile: CDS vs non-CDS by tool

In [ ]:
SC_ORDER  = ["ATG", "near_cognate", "other", "missing"]
SC_COLORS = {"ATG": "#2ca02c", "near_cognate": "#ff7f0e", "other": "#9467bd", "missing": "#cccccc"}

sc = (
    summary.groupby(["source_tool", "cls", "start_codon_class"])["n"]
    .sum()
    .reset_index()
)
sc["pct"] = (
    100 * sc["n"] /
    sc.groupby(["source_tool", "cls"])["n"].transform("sum")
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
x = np.arange(len(TOOL_ORDER))

for ax, cls in zip(axes, CLS_ORDER):
    subset = (
        sc[sc.cls == cls]
        .pivot_table(index="source_tool", columns="start_codon_class",
                     values="pct", fill_value=0)
        .reindex(index=TOOL_ORDER)
        .reindex(columns=SC_ORDER, fill_value=0)
    )
    bottom = np.zeros(len(TOOL_ORDER))
    for sc_type in SC_ORDER:
        vals = subset[sc_type].values if sc_type in subset.columns else np.zeros(len(TOOL_ORDER))
        ax.bar(x, vals, bottom=bottom, color=SC_COLORS[sc_type], label=sc_type, alpha=0.9)
        bottom += vals
    ax.set_xticks(x); ax.set_xticklabels(TOOL_ORDER, rotation=0)
    ax.set_ylabel("% of ORFs" if cls == CLS_ORDER[0] else "")
    ax.set_ylim(0, 105)
    ax.set_title(f"Start codon profile — {CLS_LABELS[cls]}")
    if cls == CLS_ORDER[0]:
        ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.savefig("all_tool_start_codon_profile.png", dpi=180, bbox_inches="tight")
plt.show()


## Plot 5 — ORF size distribution by tool and class

In [ ]:
# Splice length distribution (nt) — aggregate from summary
# Load a sample of translons for density plotting
size_df = pd.read_sql("""
    SELECT source_tool, source_feature_class AS cls, spliced_length_nt
    FROM translons
    WHERE qc_status = 'pass'
      AND sample_id NOT GLOB '*_fastq'
      AND source_feature_class IN ('cds', 'non_cds')
""", con)

fig, axes = plt.subplots(2, len(TOOL_ORDER), figsize=(16, 7), sharey="row")
bins = np.logspace(np.log10(30), np.log10(20000), 50)

for col, tool in enumerate(TOOL_ORDER):
    for row, cls in enumerate(CLS_ORDER):
        ax = axes[row][col]
        vals = size_df.loc[
            (size_df["source_tool"] == tool) & (size_df["cls"] == cls),
            "spliced_length_nt"
        ].values
        if len(vals) == 0:
            ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        else:
            ax.hist(vals, bins=bins, color=CLS_COLORS[cls], alpha=0.8, density=True)
            ax.axvline(np.median(vals), color="black", lw=1.2, ls="--",
                       label=f"median={np.median(vals):.0f}nt")
            ax.legend(fontsize=7)
        ax.set_xscale("log")
        if col == 0:
            ax.set_ylabel(CLS_LABELS[cls])
        if row == 0:
            ax.set_title(tool)
        if row == 1:
            ax.set_xlabel("Spliced length (nt)")

fig.suptitle("ORF spliced-length distribution (log x-axis)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("all_tool_orf_size_distribution.png", dpi=180, bbox_inches="tight")
plt.show()


## Plot 6 — Cross-tool sample coverage matrix

In [ ]:
EXPECTED_SAMPLES = [
    "Fib_24_45m", "Fib_24_bsl", "Fib_27_45m", "Fib_27_bsl", "Fib_41_45m", "Fib_41_bsl",
    "Ribo_Fib_pooled",
    "SRR15513179", "SRR15513180", "SRR15513181", "SRR15513182",
    "SRR15513197", "SRR15513198_GENELAB1026", "SRR15513199",
    "SRR15513200", "SRR15513201", "SRR15513202_GENELAB1143",
    "Ribo_EC_pooled",
    "SRR11005875_to_79", "SRR11005880_to_84", "SRR11005885_to_89",
    "SRR11005890_to_94", "SRR11005895_to_99", "SRR11005900_to_04",
    "Ribo_Pancreas_pooled",
]

coverage = pd.read_sql(
    "SELECT source_tool, sample_id, source_feature_class AS cls, COUNT(*) AS n "
    "FROM translons WHERE source_feature_class IN ('cds','non_cds') "
    "GROUP BY source_tool, sample_id, source_feature_class",
    con,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for ax, cls in zip(axes, CLS_ORDER):
    mat = (
        coverage[coverage.cls == cls]
        .pivot_table(index="sample_id", columns="source_tool", values="n", fill_value=0)
        .reindex(index=EXPECTED_SAMPLES, columns=TOOL_ORDER, fill_value=0)
    )
    im = ax.imshow(np.log10(mat.values + 1), aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(len(TOOL_ORDER))); ax.set_xticklabels(TOOL_ORDER, rotation=30, ha="right")
    ax.set_yticks(range(len(EXPECTED_SAMPLES))); ax.set_yticklabels(EXPECTED_SAMPLES, fontsize=7)
    plt.colorbar(im, ax=ax, label="log10(ORFs + 1)")
    ax.set_title(f"Sample × Tool coverage\n({CLS_LABELS[cls]})")

plt.tight_layout()
plt.savefig("all_tool_sample_coverage_heatmap.png", dpi=180, bbox_inches="tight")
plt.show()


## Summary table

In [ ]:
summary_table = (
    summary.groupby(["source_tool", "cls"])["n"]
    .sum()
    .unstack(fill_value=0)
    .reindex(index=TOOL_ORDER, columns=CLS_ORDER, fill_value=0)
)
summary_table["total"] = summary_table.sum(axis=1)
summary_table["cds_fraction"] = (
    summary_table.get("cds", 0) / summary_table["total"].replace(0, np.nan) * 100
).round(1)
display(summary_table)
